In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image in grayscale
uploaded = cv2.imread('/content/slynerd.jpg')

# Gray
gray = cv2.cvtColor(uploaded, cv2.COLOR_BGR2GRAY)

# Apply fixed threshold
_, fixed_thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Apply adaptive threshold
adaptive_thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                        cv2.THRESH_BINARY, 11, 2)

# Process thresholded image
def process(thresh_img, original_img, title=""):
    contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    output = original_img.copy()
    total_area = 0
    total_perimeter = 0

    for cnt in contours:
        # Draw contour
        cv2.drawContours(output, [cnt], -1, (0, 255, 0), 2)

        # Moments and center of mass
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            cv2.circle(output, (cx, cy), 4, (0, 0, 255), -1)

        # Bounding box
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(output, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Metrics
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        total_area += area
        total_perimeter += perimeter

    num_contours = len(contours)
    avg_area = total_area / num_contours if num_contours > 0 else 0
    avg_perimeter = total_perimeter / num_contours if num_contours > 0 else 0

    print(f"{title} Threshold")
    print(f"Number of detected forms: {num_contours}")
    print(f"Average Area: {avg_area:.2f}")
    print(f"Average Perimeter: {avg_perimeter:.2f}\n")

    # Convert BGR to RGB for display with matplotlib
    output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
    return output_rgb

# Process both thresholded images
output_fixed = process(fixed_thresh, uploaded, "Fixed")
output_adaptive = process(adaptive_thresh, uploaded, "Adaptive")

# Show results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title('Fixed Threshold')
plt.imshow(output_fixed)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Adaptive Threshold')
plt.imshow(output_adaptive)
plt.axis('off')

plt.tight_layout()
plt.show()